# W1-10: packaged Jais adapter smoke check
Use a GPU runtime. This notebook checks the packaged module on the existing
five-clip Task 10 smoke selection. It can reuse the pinned model already in
memory. Shared research split/pilot selection remains Task 7 work.

Run each cell in order. The source checkout, input hashes, direct dependencies,
cache counts and measurements are recorded. The five clips are not a benchmark.


In [ ]:
import json, subprocess, sys, tempfile, time, hashlib
from pathlib import Path
from datetime import datetime, timezone
SOURCE_REF = 'eec-14-jais-adapter'
REPO_ROOT = Path(tempfile.mkdtemp(prefix='jais-module-'))
subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--filter=blob:none',
                '--sparse', '--branch', SOURCE_REF,
                'https://github.com/alizbeeb5/Arabic-Dialect-Identification.git',
                str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'sparse-checkout', 'set',
                'src/models', 'src/evaluation', 'configs', 'environments'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(REPO_ROOT / 'environments/jais.txt')], check=True)
sys.path.insert(0, str(REPO_ROOT))
from src.models.jais_adapter import (
    JaisAdapter, MODEL_REVISION, run_cached_jais, write_outputs, sha256_file,
)
SOURCE_COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('Adapter source:', SOURCE_COMMIT)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/Arabic-Dialect-Identification-data')
# Edit this path when using another existing five-clip Task 10 selection.
SMOKE_MANIFEST = Path('/content/drive/MyDrive/Arabic-Dialect-Identification-data/runs/jais_smoke_inputs/20261020T214046797682Z/train.jsonl')
SMOKE_METADATA = SMOKE_MANIFEST.with_suffix('.metadata.json')
metadata = json.loads(SMOKE_METADATA.read_text(encoding='utf-8'))
clips = [json.loads(line) for line in SMOKE_MANIFEST.read_text().splitlines() if line.strip()]
assert len(clips) == 5, 'Use the separate five-clip smoke selection.'
assert all(row['split'] == 'train' for row in clips)
assert sha256_file(SMOKE_MANIFEST) == metadata['manifest_sha256']
AUDIO_ROOT = Path(metadata['audio_root'])  # Override for another storage root.
CACHE_ROOT = DATA_ROOT / 'cache/jais_adapter_module_v1'
RUN_DIR = DATA_ROOT / 'runs/jais_module_smoke' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
print('Selected IDs:', [row['clip_id'] for row in clips])
print('Output:', RUN_DIR)


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime.'
if 'model' in globals() and 'processor' in globals():
    adapter = JaisAdapter(model, processor, revision=MODEL_REVISION)
else:
    adapter = JaisAdapter.from_pretrained(dtype='float16')
print('Device:', adapter.device, 'dtype:', adapter.dtype)


In [ ]:
started_at = datetime.now(timezone.utc).isoformat()
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
records, counts = run_cached_jais(clips, AUDIO_ROOT, CACHE_ROOT, adapter, batch_size=2)
torch.cuda.synchronize()
wall_seconds = time.perf_counter() - started
run_record = {
    'experiment': 'task10_packaged_module_smoke', 'git_commit': SOURCE_COMMIT,
    'adapter_code_sha256': sha256_file(REPO_ROOT / 'src/models/jais_adapter.py'),
    'model_revision': MODEL_REVISION, 'inference_config': adapter.identity,
    'dataset_version': metadata['dataset_version'], 'split_version': metadata['split_version'],
    'manifest_sha256': metadata['manifest_sha256'], 'labels_sha256': metadata['labels_sha256'],
    'clip_ids': [row['clip_id'] for row in clips],
    'started_at': started_at, 'finished_at': datetime.now(timezone.utc).isoformat(),
    'timing_scope': 'cache/audio reads, preprocessing, generation and cache writes; excludes model loading/export',
    'wall_seconds': wall_seconds, 'counts': counts,
    'peak_gpu_allocated_gib': torch.cuda.max_memory_allocated() / 2**30,
}
write_outputs(RUN_DIR, records, run_record)
print('Counts:', counts)
print('Seconds:', round(wall_seconds, 2))
print('Peak allocated GiB:', round(run_record['peak_gpu_allocated_gib'], 2))


In [ ]:
again, resume_counts = run_cached_jais(clips, AUDIO_ROOT, CACHE_ROOT, adapter, batch_size=2)
assert again == records
assert resume_counts['cache_hits'] == len(clips) and resume_counts['model_batches'] == 0
missing = {'clip_id': 'smoke:missing-audio', 'audio_path': 'missing-' + str(time.time_ns()) + '.wav'}
failed_records, failure_counts = run_cached_jais([clips[0], missing], AUDIO_ROOT, CACHE_ROOT, adapter)
assert len(failed_records) == 2 and failed_records[1]['prediction']['status'] == 'audio_error'
assert failure_counts['failed'] == 1 and failure_counts['model_batches'] == 0
print('Resume:', resume_counts)
print('Missing-audio check:', failure_counts)


In [ ]:
EVALUATION_DIR = RUN_DIR / 'evaluation'
subprocess.run([
    sys.executable, str(REPO_ROOT / 'src/evaluation/evaluate.py'),
    '--manifest', str(SMOKE_MANIFEST), '--metadata', str(SMOKE_METADATA),
    '--predictions', str(RUN_DIR / 'predictions.jsonl'),
    '--labels', str(REPO_ROOT / 'configs/labels.json'), '--split', 'train',
    '--model-name', 'Jais-ASR-packaged-smoke', '--output-dir', str(EVALUATION_DIR),
], check=True)
metrics = json.loads((EVALUATION_DIR / 'metrics.json').read_text())
print('Expected/successful:', metrics['expected'], metrics['successful'])
print('Smoke accuracy:', metrics['accuracy'], 'macro-F1 labels:', metrics['macro_f1_labels'])
print('Artifacts:', RUN_DIR)
